# Download, filter, and visualize conversational selected activations

This notebook works with the batch files produced by `scripts/run_activation_caching_conversational_selected_acts.sh`. It downloads the self-contained activation batches from the hardcoded GCS path, filters rows using their embedded prompt metadata, loads `layer_out/21` at the cached final-token position, optionally averages matching metadata groups, projects the activations with PCA, and visualizes the result.

## 1. Setup

For a fresh Colab runtime, uncomment the clone and authentication commands. Local runs can skip them.

In [ ]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
%cd temporal-manifolds
# !gcloud auth application-default login
# !mv -n .env.example .env

In [ ]:
import gc
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from google.cloud import storage
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')

## 2. Configure the workflow

`METADATA_FILTERS` accepts any embedded prompt-metadata field, with dotted paths for nested fields. A scalar requires an exact match; a list keeps any listed value. Different fields are combined with AND. Use `None` or `{}` to disable filtering. `AGG_BY` lists flattened metadata fields whose equal values define averaging groups; the derived `time_horizon_months` field is supported. Use `None` or `[]` to retain individual rows.

In [ ]:
METADATA_FILTERS: dict[str, object] | None = {
    'template_metadata.prompt_framing': 'task_available_time',
    # 'task_metadata.domain': 'communication',
    # 'template_metadata.output_format': 'approach_actions',
}

# Output contract of scripts/run_activation_caching_conversational_selected_acts.sh.
ACTIVATIONS_GCS_URI = 'gs://temporal-research-bucket/selected_acts'
DATA_DIR = repo_root / 'data' / 'filtered_conversational_selected_activations'
PROJECT_ID = os.getenv('GCP_PROJECT_ID')
OVERWRITE = False
MAX_SAMPLES: int | None = None
SELECTED_BATCH_FILES: list[str] | None = None

LAYER_COMPONENT = 'layer_out/21'
POSITION_INDEX = 0  # The only cached position; payload position value is -1.
AGG_BY: list[str] | None = ['time_horizon_months']
PCA_MODEL_LOAD_PATH: Path | None = None
PCA_MODEL_SAVE_PATH: Path | None = None  # For example: DATA_DIR / 'pca_model.pkl'
PHRASING_FIELDS = ['template_metadata.prompt_framing', 'template_metadata.output_format']

## 3. Download activation batches

All `activations_batch_*.pt` objects below the hardcoded bucket path are cached locally and skipped on later runs unless `OVERWRITE` is true.

In [ ]:
def parse_gcs_uri(uri):
    if not uri.startswith('gs://'):
        raise ValueError(f'Expected a gs:// URI, got {uri!r}')
    bucket, separator, prefix = uri[5:].partition('/')
    if not bucket or not separator or not prefix.strip('/'):
        raise ValueError(f'GCS URI must contain a bucket and prefix: {uri!r}')
    return bucket, prefix.strip('/')


client = storage.Client(project=PROJECT_ID)
bucket_name, activation_prefix = parse_gcs_uri(ACTIVATIONS_GCS_URI)
blobs = sorted(
    (
        blob for blob in client.bucket(bucket_name).list_blobs(prefix=activation_prefix.rstrip('/') + '/')
        if Path(blob.name).name.startswith('activations_batch_') and blob.name.endswith('.pt')
    ),
    key=lambda blob: blob.name,
)
if not blobs:
    raise FileNotFoundError(f'No activation batches found below {ACTIVATIONS_GCS_URI}')

batch_dir = DATA_DIR / 'batches'
batch_dir.mkdir(parents=True, exist_ok=True)
downloaded_batch_paths = []
for blob in tqdm(blobs, desc='Downloading batches'):
    destination = batch_dir / Path(blob.name).name
    if OVERWRITE or not destination.exists():
        blob.download_to_filename(str(destination))
    downloaded_batch_paths.append(destination)

print(f'GCS source: {ACTIVATIONS_GCS_URI}')
print(f'Downloaded/local batches: {len(downloaded_batch_paths):,}')

### 3.1 Select local batch files

Set `SELECTED_BATCH_FILES` to basenames such as `activations_batch_00000.pt`, or leave it as `None` to read every downloaded batch.

In [ ]:
downloaded_by_name = {path.name: path for path in downloaded_batch_paths}
if SELECTED_BATCH_FILES is None:
    selected_batch_paths = downloaded_batch_paths
else:
    missing_files = sorted(set(SELECTED_BATCH_FILES) - set(downloaded_by_name))
    if missing_files:
        raise FileNotFoundError(f'Selected batch files were not downloaded: {missing_files}')
    selected_batch_paths = [downloaded_by_name[name] for name in SELECTED_BATCH_FILES]

if not selected_batch_paths:
    raise ValueError('No local batch files were selected.')
print(f'Selected {len(selected_batch_paths):,} of {len(downloaded_batch_paths):,} local batches.')
for path in selected_batch_paths[:10]:
    print(path)

## 4. Filter metadata and load activations

Each batch already contains aligned `sample_indices`, `prompts`, `prompt_metadata`, and activations. The cell validates that contract, selects matching rows, and concatenates only `layer_out/21` at its sole cached position.

In [ ]:
_MISSING = object()


def get_metadata_field(metadata, dotted_path):
    value = metadata
    for key in dotted_path.split('.'):
        if not isinstance(value, dict) or key not in value:
            return _MISSING
        value = value[key]
    return value


def metadata_value_matches(actual_value, expected_value):
    if actual_value is _MISSING:
        return False
    allowed_values = expected_value if isinstance(expected_value, list) else [expected_value]
    return any(actual_value == allowed_value for allowed_value in allowed_values)


def load_batch(path):
    return torch.load(path, map_location='cpu', weights_only=True, mmap=True)


feature_parts = []
loaded_indices = []
loaded_prompts = []
loaded_metadata = []
cached_position = None
for path in tqdm(selected_batch_paths, desc='Filtering activation batches'):
    payload = load_batch(path)
    sample_indices = payload['sample_indices']
    prompts = payload['prompts']
    metadata_rows = payload['prompt_metadata']
    tensor = payload['activations'][LAYER_COMPONENT]
    if not (len(sample_indices) == len(prompts) == len(metadata_rows) == tensor.shape[0]):
        raise ValueError(f'Misaligned rows in {path}')
    positions = list(payload['positions'])
    if POSITION_INDEX >= len(positions):
        raise ValueError(f'{path} has only {len(positions)} cached positions.')
    position_value = positions[POSITION_INDEX]
    if cached_position is None:
        cached_position = position_value
    elif position_value != cached_position:
        raise ValueError(f'Inconsistent cached positions in {path}: {positions}')
    row_offsets = [
        offset for offset, metadata in enumerate(metadata_rows)
        if all(
            metadata_value_matches(get_metadata_field(metadata, field), expected)
            for field, expected in (METADATA_FILTERS or {}).items()
        )
    ]
    if MAX_SAMPLES is not None:
        remaining = MAX_SAMPLES - len(loaded_indices)
        row_offsets = row_offsets[:max(remaining, 0)]
    if row_offsets:
        rows = torch.as_tensor(row_offsets, dtype=torch.long)
        feature_parts.append(tensor[:, POSITION_INDEX, :].index_select(0, rows).to(torch.float32))
        loaded_indices.extend(int(sample_indices[offset]) for offset in row_offsets)
        loaded_prompts.extend(prompts[offset] for offset in row_offsets)
        loaded_metadata.extend(metadata_rows[offset] for offset in row_offsets)
    del tensor, payload
    if MAX_SAMPLES is not None and len(loaded_indices) >= MAX_SAMPLES:
        break

if not feature_parts:
    raise ValueError('No activation rows matched the configured filters.')
activation_matrix = torch.cat(feature_parts, dim=0)
del feature_parts
gc.collect()
print('Activation matrix shape:', tuple(activation_matrix.shape))
print(f'Loaded {len(loaded_indices):,} samples from {LAYER_COMPONENT} at position {cached_position}.')
print('First sample indices:', loaded_indices[:10])

## 5. Prepare metadata and project with PCA

Scalar metadata is flattened to dotted columns. Equivalent durations are normalized to `time_horizon_months` before optional grouping. A new three-component PCA is fitted unless `PCA_MODEL_LOAD_PATH` points to a trusted saved model.

In [ ]:
def flatten_scalar_metadata(metadata, prefix=''):
    flattened = {}
    for key, value in metadata.items():
        path = f'{prefix}.{key}' if prefix else key
        if isinstance(value, dict):
            flattened.update(flatten_scalar_metadata(value, path))
        elif not isinstance(value, (list, tuple, set)):
            flattened[path] = value
    return flattened


metadata_df = pd.DataFrame([flatten_scalar_metadata(metadata) for metadata in loaded_metadata])
metadata_df.insert(0, 'sample_index', loaded_indices)
metadata_df.insert(1, 'prompt', loaded_prompts)
unit_to_months = {
    'second': 1 / (30.4375 * 86400), 'minute': 1 / (30.4375 * 1440),
    'hour': 1 / (30.4375 * 24), 'day': 1 / 30.4375, 'week': 7 / 30.4375,
    'month': 1, 'year': 12, 'decade': 120, 'century': 1200, 'millennium': 12000,
}
unit_to_months.update({f'{unit}s': value for unit, value in list(unit_to_months.items())})
unit_to_months['centuries'] = 1200
unit_to_months['millennia'] = 12000
value_field = 'base_value' if 'base_value' in metadata_df else 'value'
unit_field = 'base_unit' if 'base_unit' in metadata_df else 'unit'
if value_field in metadata_df and unit_field in metadata_df:
    unknown_units = sorted(set(metadata_df[unit_field].astype(str).str.lower()) - set(unit_to_months))
    if unknown_units:
        raise ValueError(f'Cannot convert time-horizon units to months: {unknown_units}')
    metadata_df['time_horizon_months'] = [
        round(float(value) * unit_to_months[str(unit).lower()], 12)
        for value, unit in zip(metadata_df[value_field], metadata_df[unit_field])
    ]

aggregation_fields = list(AGG_BY or [])
missing_aggregation_fields = set(aggregation_fields) - set(metadata_df.columns)
if missing_aggregation_fields:
    raise ValueError(f'Missing AGG_BY fields: {sorted(missing_aggregation_fields)}')
if len(set(aggregation_fields)) != len(aggregation_fields):
    raise ValueError('AGG_BY must not contain duplicate fields.')
available_phrasing_fields = [field for field in PHRASING_FIELDS if field in metadata_df]

if aggregation_fields:
    vectors, rows = [], []
    for row_offsets in metadata_df.groupby(aggregation_fields, dropna=False, sort=False).indices.values():
        row_offsets = list(row_offsets)
        vectors.append(activation_matrix.index_select(0, torch.as_tensor(row_offsets)).mean(dim=0))
        row = metadata_df.iloc[row_offsets[0]].copy()
        row['source_sample_count'] = len(row_offsets)
        for field in [value_field, unit_field, *available_phrasing_fields]:
            if field in metadata_df and metadata_df.iloc[row_offsets][field].nunique(dropna=True) > 1:
                row[field] = '<averaged>'
        rows.append(row)
    pca_matrix = torch.stack(vectors)
    analysis_metadata_df = pd.DataFrame(rows).reset_index(drop=True)
else:
    pca_matrix = activation_matrix
    analysis_metadata_df = metadata_df.copy()
    analysis_metadata_df['source_sample_count'] = 1

pca_input = pca_matrix.cpu().numpy()
if PCA_MODEL_LOAD_PATH is None:
    if min(pca_input.shape) < 3:
        raise ValueError(f'At least three rows and features are required, got {pca_input.shape}.')
    pca = PCA(n_components=3, svd_solver='randomized', random_state=0)
    projections = pca.fit_transform(pca_input)
    print('Fitted PCA model on the prepared activation matrix.')
else:
    with Path(PCA_MODEL_LOAD_PATH).open('rb') as model_file:
        pca = pickle.load(model_file)
    if not isinstance(pca, PCA):
        raise TypeError('PCA_MODEL_LOAD_PATH does not contain a scikit-learn PCA model.')
    projections = pca.transform(pca_input)

if projections.shape[1] != 3:
    raise ValueError(f'PCA model must produce exactly 3 components, got {projections.shape[1]}.')
if PCA_MODEL_SAVE_PATH is not None:
    save_path = Path(PCA_MODEL_SAVE_PATH)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with save_path.open('wb') as model_file:
        pickle.dump(pca, model_file)
    print(f'Saved PCA model to {save_path}.')

print('PCA input shape:', tuple(pca_matrix.shape))
print('Explained variance fractions:', pca.explained_variance_ratio_)

## 6. Visualize the projection

The interactive controls discover flattened scalar metadata fields automatically and update a 3D PCA scatter plot.

In [ ]:
df_projs = analysis_metadata_df.copy().reset_index(drop=True)
df_projs[['PC1', 'PC2', 'PC3']] = projections
if 'time_horizon_months' in df_projs:
    if (df_projs['time_horizon_months'] <= 0).any():
        raise ValueError('Time horizons must be positive before taking the logarithm.')
    df_projs['log10_time_horizon_months'] = np.log10(df_projs['time_horizon_months'])
excluded_fields = {'PC1', 'PC2', 'PC3', 'sample_index', 'prompt'}
metadata_fields = sorted(column for column in df_projs if column not in excluded_fields)
preferred_color = 'log10_time_horizon_months' if 'log10_time_horizon_months' in df_projs else metadata_fields[0]
color_fields = [preferred_color, *[field for field in metadata_fields if field != preferred_color]]
print(f'Prepared {len(df_projs):,} projected points.')

In [ ]:
import ipywidgets as widgets
import plotly.express as px
import plotly.io as pio
from IPython.display import display
from pandas.api.types import is_bool_dtype, is_numeric_dtype

try:
    from google.colab import output as colab_output
except ImportError:
    in_colab = False
else:
    in_colab = True
    colab_output.enable_custom_widget_manager()
    pio.renderers.default = 'colab'

color_dropdown = widgets.Dropdown(options=color_fields, value=preferred_color, description='Color by:', layout=widgets.Layout(width='500px'))
filter_dropdown = widgets.Dropdown(options=[('(no filter)', None), *[(field, field) for field in metadata_fields]], description='Filter by:', layout=widgets.Layout(width='500px'))
filter_values = widgets.SelectMultiple(description='Keep:', rows=6, layout=widgets.Layout(width='500px'))
plot_output = widgets.Output()

def update_filter_values():
    field = filter_dropdown.value
    values = [] if field is None else sorted(df_projs[field].dropna().unique().tolist(), key=str)
    filter_values.options = [(str(value), value) for value in values]
    filter_values.value = tuple(values)
    filter_values.disabled = field is None

def render_plot(change=None):
    filtered = df_projs
    if filter_dropdown.value is not None:
        filtered = filtered[filtered[filter_dropdown.value].isin(filter_values.value)]
    plot_output.clear_output(wait=True)
    with plot_output:
        if filtered.empty:
            print('No points match the selected filter values.')
            return
        color_field = color_dropdown.value
        plot_data = filtered.copy()
        numeric_color = is_numeric_dtype(plot_data[color_field]) and not is_bool_dtype(plot_data[color_field])
        if not numeric_color:
            plot_data[color_field] = plot_data[color_field].astype('string').fillna('<missing>')
        hover_fields = ['sample_index', 'prompt']
        if 'time_horizon_months' in plot_data:
            hover_fields.append('time_horizon_months')
        fig = px.scatter_3d(
            plot_data, x='PC1', y='PC2', z='PC3', color=color_field,
            color_continuous_scale='Viridis' if numeric_color else None,
            hover_data=hover_fields,
            title=f'{LAYER_COMPONENT}, cached position {cached_position} ({len(plot_data):,} points)',
            opacity=0.7,
        )
        fig.update_traces(marker={'size': 4})
        fig.show(renderer='colab') if in_colab else display(fig)

def on_filter_change(change):
    update_filter_values()
    render_plot()

color_dropdown.observe(render_plot, names='value')
filter_dropdown.observe(on_filter_change, names='value')
filter_values.observe(render_plot, names='value')
update_filter_values()
display(widgets.VBox([color_dropdown, filter_dropdown, filter_values, plot_output]))
render_plot()

In [ ]:
df_projs